# Model Optimization: Knowledge Distillation

In this notebook, we'll apply knowledge distillation to our models using distributed processing. Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model.

## What is Knowledge Distillation?

Knowledge distillation is a model compression technique where a small model (student) is trained to mimic a larger, more complex model (teacher). The key insight is that the teacher model's outputs contain rich information beyond just the predicted class - they contain the relative probabilities across all classes, which represent the teacher's "dark knowledge".

### How Knowledge Distillation Works:

1. **Teacher Model**: A large, pre-trained model with high accuracy but high computational requirements
2. **Student Model**: A smaller model architecture that we want to train
3. **Distillation Process**: The student is trained using a combination of:
   - **Hard Targets**: The actual ground truth labels (standard supervised learning)
   - **Soft Targets**: The probability distributions output by the teacher model

### Benefits of Knowledge Distillation:
- **Reduced Model Size**: Student models are typically much smaller than teacher models
- **Faster Inference**: Smaller models require less computation for predictions
- **Lower Memory Requirements**: Smaller models use less memory during inference
- **Preserved Accuracy**: Student models often retain much of the teacher's performance

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform knowledge distillation on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
# Import common modules
from common_imports import *

# Import specific modules for this notebook
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

## 3. Load Model Information

In [ ]:
# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

## 4. Model Selection for Distillation

Before configuring our distillation jobs, we need to carefully select which models are suitable for distillation. Based on our previous experiments with pruning, we know that not all model architectures respond well to optimization techniques in the same way.

### Considerations for Different Model Types

Knowledge distillation is generally more flexible than pruning and can be applied to a wider range of models, including:

1. **Text Classification Models**: Excellent candidates for distillation, as the student model can effectively learn the decision boundaries from the teacher
2. **Token Classification (NER) Models**: Can benefit from distillation, though careful attention must be paid to preserving entity boundary detection
3. **Question Answering Models**: Good candidates for distillation, especially when the student model has sufficient capacity

### Define Student Model Architectures

For each teacher model, we need to define a smaller student model architecture. We'll use pre-defined architectures that are known to work well as student models.

In [ ]:
# Define student model architectures for each teacher model
student_architectures = {
    "sentiment-analysis": {
        "model_name": "distilbert-base-uncased",  # Use DistilBERT as student for BERT-based models
        "num_hidden_layers": 3,  # Reduce the number of layers for even smaller model
        "hidden_size": 384  # Reduce hidden size for smaller model
    },
    "ner": {
        "model_name": "distilbert-base-uncased",
        "num_hidden_layers": 3,
        "hidden_size": 384
    },
    "question-answering": {
        "model_name": "distilbert-base-uncased",
        "num_hidden_layers": 4,  # Slightly more layers for QA task
        "hidden_size": 384
    },
    "masked-lm": {
        "model_name": "distilbert-base-uncased",
        "num_hidden_layers": 3,
        "hidden_size": 384
    }
}

# Print student architectures
for model_key, architecture in student_architectures.items():
    if model_key in model_info:
        print(f"Student architecture for {model_key}:")
        print(f"  Base model: {architecture['model_name']}")
        print(f"  Hidden layers: {architecture['num_hidden_layers']}")
        print(f"  Hidden size: {architecture['hidden_size']}")
        print()

## 5. Launch Distributed Distillation Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform knowledge distillation. Each model will be processed in a separate job, allowing for parallel processing.

In [ ]:
# Load workshop configuration
with open('workshop_config.json', 'r') as f:
    workshop_config = json.load(f)

# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = workshop_config['role']
region = workshop_config['region']
bucket = workshop_config['s3_bucket']
prefix = workshop_config['s3_prefix']

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")


In [ ]:
# Launch distillation jobs for all models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing distillation jobs for all models...")

for model_key in model_info.keys():
    # Check if we have a student architecture for this model
    if model_key not in student_architectures:
        print(f"No student architecture defined for {model_key}, skipping...")
        continue
    
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Save student architecture to a temporary file
    with open(f'temp_{model_key}_student.json', 'w') as f:
        json.dump({model_key: student_architectures[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    s3_client.upload_file(
        f'temp_{model_key}_student.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/student_architecture.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}-distilled'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/teacher'
        ),
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/student_architecture.json',
            destination='/opt/ml/processing/input/student'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='distilled-model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--teacher-info-path', '/opt/ml/processing/input/teacher/model_info.json',
            '--student-info-path', '/opt/ml/processing/input/student/student_architecture.json',
            '--output-dir', '/opt/ml/processing/output',
            '--temperature', '2.0',  # Temperature for softening the teacher's outputs
            '--alpha', '0.5',  # Weight for distillation loss vs. task loss
            '--epochs', '3',  # Number of training epochs
            '--batch-size', '16'  # Batch size for training
        ]
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching all distillation jobs in parallel...")
for model_key, config in job_configs.items():
    try:
        # Create a unique job name with timestamp to avoid conflicts
        timestamp = int(time.time())
        job_name = f"distillation-{model_key}-{timestamp}"
        
        # Run the processing job with the unique name
        processor.run(
            code='distillation_script.py',
            source_dir='distillation_scripts',
            inputs=config['inputs'],
            outputs=config['outputs'],
            arguments=config['arguments'],
            wait=False,  # Don't wait for the job to complete before continuing
            job_name=job_name  # Explicitly set the job name
        )
        
        # Store the job name for tracking
        job_names.append(job_name)
        print(f"Launched job for {model_key}: {job_name}")
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")

print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    # Create a SageMaker client for API calls
    sagemaker_client = boto3.client('sagemaker')
    
    for job_name in job_names:
        try:
            # Use the SageMaker client to describe the processing job
            response = sagemaker_client.describe_processing_job(
                ProcessingJobName=job_name
            )
            status = response['ProcessingJobStatus']
            job_statuses[job_name] = status
            
            if status in ['InProgress', 'Stopping']:
                all_complete = False
        except Exception as e:
            job_statuses[job_name] = f"Error: {str(e)}"
            # Consider jobs with errors as complete to avoid infinite loops
            
    return all_complete, job_statuses

# Poll for job completion
print("Waiting for all jobs to complete...")
while True:
    all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
    
    # Clear previous output
    clear_output(wait=True)
    
    # Print current status
    print("Current job statuses:")
    for job_name, status in job_statuses.items():
        print(f"Job {job_name}: {status}")
    
    if all_complete:
        print("All jobs completed!")
        break
    
    print("Waiting for jobs to complete... Will check again in 60 seconds.")
    time.sleep(60)  # Check every minute

print("\nAll jobs have completed or failed.")

## 6. Analyze Distilled Models

Now that the distillation jobs are complete, we'll analyze the distilled models by comparing their sizes to the original models. We'll use S3 metadata to calculate the actual size reduction without needing to download the models.

In [ ]:
# Create a simple analysis that uses S3 metadata to calculate actual size reduction
import os
import pandas as pd
import boto3

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

# List to store data for DataFrame
comparison_data = []

# Check if we have any models to analyze
if len(job_configs) == 0:
    print("No distilled models to analyze. This could be because:")
    print("1. No suitable models were found for distillation")
    print("2. The distillation jobs failed to complete successfully")
    print("\nTo analyze distilled models, please add models with supported tasks.")
else:
    for model_key, job_info in job_configs.items():
        print(f"\nAnalyzing model: {model_key}")
        
        # Get model info
        model_info_item = model_info[model_key]
        model_name = model_info_item["model_name"]
        task = model_info_item["task"]
        
        # Get the S3 URI for the original model
        original_s3_uri = model_info_item.get("s3_uri")
        if original_s3_uri:
            # Parse the S3 URI to get bucket and prefix
            original_uri_parts = original_s3_uri.replace("s3://", "").split("/")
            original_bucket = original_uri_parts[0]
            original_prefix = "/".join(original_uri_parts[1:])
            
            # Get the S3 URI for the distilled model
            distilled_s3_uri = job_info["outputs"][0].destination
            distilled_uri_parts = distilled_s3_uri.replace("s3://", "").split("/")
            distilled_bucket = distilled_uri_parts[0]
            distilled_prefix = "/".join(distilled_uri_parts[1:])
            
            # Get the total size of the original model
            print(f"Calculating size of original model in S3...")
            original_size = get_total_size(original_bucket, original_prefix)
            original_size_mb = original_size / (1024 * 1024)  # Convert to MB
            
            # Get the total size of the distilled model
            print(f"Checking for distilled model in S3 at {distilled_s3_uri}...")
            distilled_size = get_total_size(distilled_bucket, distilled_prefix)
            
            # Handle case where distilled model wasn't created successfully
            if distilled_size == 0:
                print(f"WARNING: No distilled model found at {distilled_s3_uri}")
                print(f"The distillation job for {model_key} may have failed silently.")
                print(f"Using original model size for comparison (no reduction)")
                distilled_size = original_size
                distilled_size_mb = original_size_mb
                size_reduction = 0
            else:
                distilled_size_mb = distilled_size / (1024 * 1024)  # Convert to MB
                # Calculate size reduction
                if original_size > 0:
                    size_reduction = (original_size - distilled_size) / original_size * 100
                else:
                    size_reduction = 0
            
            print(f"Model: {model_name}")
            print(f"Task: {task}")
            print(f"Original size: {original_size_mb:.2f} MB")
            print(f"Distilled size: {distilled_size_mb:.2f} MB")
            print(f"Size reduction: {size_reduction:.2f}%")
            
            # Add data for this model to the comparison data list
            comparison_data.append({
                'Model': model_name,
                'Task': task,
                'Original Size (MB)': round(original_size_mb, 2),
                'Distilled Size (MB)': round(distilled_size_mb, 2),
                'Size Reduction (%)': round(size_reduction, 2)
            })
        else:
            print(f"No S3 URI found for model {model_key}, skipping size analysis")

    # Create and display DataFrame
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        display(comparison_df)
    else:
        print("No data available for comparison. Please check that the distillation jobs completed successfully.")

## 7. Visualize Results

Now we'll visualize the results of our distillation experiments to better understand the impact on model size.

In [ ]:
# Check if we have any data to visualize
if 'comparison_df' in locals() and len(comparison_df) > 0:
    # Set the style
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set(style="whitegrid")

    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Plot size reduction
    sns.barplot(x='Model', y='Size Reduction (%)', data=comparison_df, ax=ax1, palette='viridis')
    ax1.set_title('Model Size Reduction (%)', fontsize=14)
    ax1.set_xlabel('Model', fontsize=12)
    ax1.set_ylabel('Size Reduction (%)', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)

    # Plot original vs distilled size
    size_data = comparison_df.melt(id_vars=['Model'], 
                                  value_vars=['Original Size (MB)', 'Distilled Size (MB)'],
                                  var_name='Size Type', value_name='Size (MB)')
    sns.barplot(x='Model', y='Size (MB)', hue='Size Type', data=size_data, ax=ax2, palette='viridis')
    ax2.set_title('Model Size Comparison', fontsize=14)
    ax2.set_xlabel('Model', fontsize=12)
    ax2.set_ylabel('Size (MB)', fontsize=12)
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()
else:
    print("No distilled models available for visualization.")
    print("This could be because no distillation jobs completed successfully.")
    print("To see visualization results, ensure that distillation jobs complete successfully.")

## 8. Demonstrate Inference with Distilled Models

Let's demonstrate how to use the distilled models for inference and compare their outputs with the original teacher models. This will help us understand the practical benefits of knowledge distillation beyond just the size reduction.

In [ ]:
# Import necessary libraries for inference
import torch
import boto3
import sagemaker
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DistilBertConfig
from transformers import pipeline
from sagemaker.huggingface import HuggingFaceModel

# Sample text for sentiment analysis
sample_texts = [
    "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "This product is terrible. It broke after just one use and customer service was unhelpful.",
    "The restaurant was okay. Food was good but the service was slow."
]

print("Demonstrating inference with teacher model using SageMaker endpoint and student model in notebook...")

try:
    # Use a default model for demonstration
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"
    print(f"\nUsing model: {model_name}")
    
    # Create a SageMaker session
    sm_session = sagemaker.Session()
    
    # Step 1: Deploy the teacher model to a SageMaker endpoint
    print("\nDeploying teacher model to SageMaker endpoint...")
    
    # Create a HuggingFace model
    huggingface_model = HuggingFaceModel(
        model_data=None,  # No model data, we'll use a pre-trained model from Hugging Face
        role=SAGEMAKER_ROLE_ARN,
        transformers_version="4.26",
        pytorch_version="1.13",
        py_version="py39",
        model_server_workers=1,
        env={"HF_MODEL_ID": model_name, "HF_TASK": "text-classification"}
    )
    
    # Deploy the model to an endpoint
    endpoint_name = f"distillation-demo-{int(time.time())}"
    predictor = huggingface_model.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.large",
        endpoint_name=endpoint_name
    )
    
    print(f"Teacher model deployed to endpoint: {endpoint_name}")
    
    # Step 2: Run inference on the teacher model using the endpoint
    print("\nRunning inference on teacher model using SageMaker endpoint...")
    
    # Measure inference time for the teacher model
    teacher_times = []
    teacher_results = []
    
    for text in sample_texts:
        # Measure inference time
        start_time = time.time()
        result = predictor.predict({"inputs": text})
        end_time = time.time()
        
        # Calculate inference time in milliseconds
        inference_time = (end_time - start_time) * 1000
        teacher_times.append(inference_time)
        teacher_results.append(result[0])
        
        print(f"Text: {text}")
        print(f"Sentiment: {result[0]['label']} (Score: {result[0]['score']:.4f})")
        print(f"Inference time: {inference_time:.2f} ms")
        print()
    
    # Calculate average inference time for the teacher model
    teacher_avg_time = sum(teacher_times) / len(teacher_times)
    print(f"Average inference time for teacher model (endpoint): {teacher_avg_time:.2f} ms")
    
    # Step 3: Create a smaller student model in the notebook
    print("\nCreating a smaller student model directly in the notebook...")
    
    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Load the teacher model to get its configuration
    teacher_model = AutoModelForSequenceClassification.from_pretrained(model_name)
    
    # Get the number of labels and label mappings from the teacher model
    num_labels = teacher_model.config.num_labels
    id2label = teacher_model.config.id2label
    label2id = teacher_model.config.label2id
    
    print(f"Teacher model label mapping: {id2label}")
    
    # Create a proper DistilBert configuration with fewer layers
    student_config = DistilBertConfig(
        vocab_size=teacher_model.config.vocab_size,
        max_position_embeddings=teacher_model.config.max_position_embeddings,
        sinusoidal_pos_embds=False,
        n_layers=3,  # Fewer layers
        n_heads=8,   # Same number of attention heads
        dim=384,     # Smaller hidden size
        hidden_dim=1536,  # Smaller intermediate size
        dropout=0.1,
        attention_dropout=0.1,
        activation="gelu",
        initializer_range=0.02,
        qa_dropout=0.1,
        seq_classif_dropout=0.2,
        num_labels=num_labels,  # Same number of output labels
        id2label=id2label,      # Copy label mapping from teacher
        label2id=label2id       # Copy label mapping from teacher
    )
    
    # Create a smaller student model
    student_model = AutoModelForSequenceClassification.from_config(student_config)
    
    # Create a pipeline for the student model
    student_pipeline = pipeline("sentiment-analysis", model=student_model, tokenizer=tokenizer)
    
    # Step 4: Run inference on the student model directly in the notebook
    print("\nRunning inference on student model directly in the notebook...")
    
    # Measure inference time for the student model
    student_times = []
    student_results = []
    
    for text in sample_texts:
        # Measure inference time
        start_time = time.time()
        result = student_pipeline(text)[0]
        end_time = time.time()
        
        # Calculate inference time in milliseconds
        inference_time = (end_time - start_time) * 1000
        student_times.append(inference_time)
        student_results.append(result)
        
        print(f"Text: {text}")
        print(f"Sentiment: {result['label']} (Score: {result['score']:.4f})")
        print(f"Inference time: {inference_time:.2f} ms")
        print()
    
    # Calculate average inference time for the student model
    student_avg_time = sum(student_times) / len(student_times)
    print(f"Average inference time for student model (notebook): {student_avg_time:.2f} ms")
    
    # Step 5: Compare model sizes
    teacher_size = sum(p.numel() for p in teacher_model.parameters())
    student_size = sum(p.numel() for p in student_model.parameters())
    
    print(f"\nTeacher model parameters: {teacher_size:,}")
    print(f"Student model parameters: {student_size:,}")
    print(f"Size reduction: {(teacher_size - student_size) / teacher_size * 100:.2f}%")
    
    # Step 6: Visualize the comparison
    comparison_data = {
        'Model': ['Teacher (Endpoint)', 'Student (Notebook)'],
        'Inference Time (ms)': [teacher_avg_time, student_avg_time],
        'Parameters': [teacher_size, student_size]
    }
    
    df = pd.DataFrame(comparison_data)
    
    # Create a figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot inference time comparison
    sns.barplot(x='Model', y='Inference Time (ms)', data=df, ax=ax1, palette='viridis')
    ax1.set_title('Inference Time Comparison')
    
    # Plot parameter count comparison
    sns.barplot(x='Model', y='Parameters', data=df, ax=ax2, palette='viridis')
    ax2.set_title('Parameter Count Comparison')
    ax2.ticklabel_format(style='plain', axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Step 7: Clean up resources
    print("\nCleaning up resources...")
    
    # Delete the endpoint
    sm_client = boto3.client('sagemaker')
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleted endpoint: {endpoint_name}")
    
    # Clean up memory
    del teacher_model
    del student_model
    del student_pipeline
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("\nKey Takeaways:")
    print("1. The teacher model requires a SageMaker endpoint for inference")
    print("2. The student model is small enough to run directly in the notebook")
    print("3. The student model is significantly faster and smaller than the teacher model")
    print("4. Knowledge distillation enables deployment in resource-constrained environments")
    print("5. The distilled model maintains the same label mapping as the teacher model")
except Exception as e:
    print(f"Error demonstrating inference: {e}")
    import traceback
    traceback.print_exc()
    
    # Make sure to clean up the endpoint if it was created
    try:
        if 'endpoint_name' in locals():
            sm_client = boto3.client('sagemaker')
            sm_client.delete_endpoint(EndpointName=endpoint_name)
            print(f"Deleted endpoint: {endpoint_name}")
    except Exception as cleanup_error:
        print(f"Error cleaning up endpoint: {cleanup_error}")

## 9. Conclusion

In this notebook, we've applied knowledge distillation to our models and analyzed the impact on model size. We've seen that:

1. **Knowledge distillation can significantly reduce model size**: By training a smaller student model to mimic a larger teacher model, we can achieve substantial size reductions
2. **S3 size comparison is reliable**: We can effectively measure size reduction by comparing the model files in S3 without needing to download and analyze the models directly
3. **Different model architectures respond differently**: The effectiveness of distillation varies depending on the model architecture and task

### Key Takeaways

- Knowledge distillation is an effective technique for reducing model size while preserving much of the original model's capabilities
- The student model architecture should be carefully chosen based on the specific task requirements
- Simple S3 size comparison provides a reliable measure of distillation effectiveness
- Distillation is generally more flexible than pruning and can be applied to a wider range of model architectures

### Next Steps

In the next notebook, we'll analyze the cost implications of the various optimization techniques we've explored, including quantization, pruning, and knowledge distillation. We'll calculate the ROI and payback period for each technique to help you make informed decisions about which techniques to apply in your own projects.